# 02-county-policy-matching

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/mobility/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('mobility'))


In [ ]:
%pprint
import numpy as np 
import pandas as pd
import os
import time
import datetime
import requests
import csv
import json
import re

import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from tqdm import tqdm

%config InlineBackend.figure_format = 'retina'
import warnings
# Keep warnings visible when checking the research environment.

In [ ]:
# Locate folder
os.chdir(workspace('mobility'))     

In [ ]:
# leavve data before day 2020-12-01
# leave_ob = (df.date-pd.to_datetime('2020-12-01')).apply(lambda x: x.days<=0)
# df = df[leave_ob]

# drop observation where subset=['admin1','admin2'] is all na
# df = df.sort_values(by=['admin1','admin2','date']).dropna(how='all',subset=['admin1','admin2']).reset_index(drop=True)

# each group number of observations
# df_group = df.groupby(by=['admin1','admin2']).apply(lambda x:len(x))

# start day and end day
# df.groupby(by=['admin1','admin2']).apply(lambda x:(list(x.date)[0],list(x.date)[-1]))

In [ ]:
def remove(df,by=['date']):
    df = df.sort_values(by=by).reset_index(drop=True)
    lim1 = (pd.to_datetime(df.date[0]) - pd.to_datetime('2020-03-01')).days > 0
    lim2 = (pd.to_datetime(df.date[len(df)-1]) - pd.to_datetime('2020-08-31')).days < 0
    lim3 = ((pd.to_datetime(df.date[len(df)-1])) - pd.to_datetime(df.date[0])).days < 180
    
    if lim1 or lim2 or lim3:
        return True
    else:
        return False      
def in_remove(ob, rmv):
    by = rmv.index.names
    
    if len(by) != 1:
        index = tuple([ob[by[i]] for i in range(len(by))])
    else:
        index = ob[by[0]]
    
    return index in rmv[rmv].index

# rmv = df.groupby(by=['sub_region_1','sub_region_2','place_id']).apply(lambda df:remove(df))
# drop_ob = df.apply(lambda ob: in_remove(ob, rmv), axis=1)
# df = df[~drop_ob]

In [ ]:
# Global Function -- Detect
def detect_Missing_days(index):
    dates = pd.to_datetime(index)
    missing_day = []
    for i in range(1,len(dates)):
        delta = (dates[i]-dates[i-1]).days
        if  delta != 1:    
            for j in range(1,delta):
                tempday = dates[i-1] + pd.Timedelta(days=j)
                daystr = tempday.strftime('%Y/%m/%d')
                missing_day.append(daystr)         
    return  missing_day    
def day_before(missing_day):
    dates = pd.to_datetime(missing_day)
    day_before = [ (date - pd.Timedelta(days=1)).strftime('%Y/%m/%d') for date in dates]
    return  day_before
def day_after(missing_day):
    dates = pd.to_datetime(missing_day)
    day_after = [ (date + pd.Timedelta(days=1)).strftime('%Y/%m/%d') for date in dates]
    return  day_after
def fill_Missing_days1(ser):
    # ser here is in df form
    dates = ser.index
    missing_day = detect_Missing_days(dates)
    if len(missing_day)==0:
           return ser
    before = day_before(missing_day)
    after  = day_after(missing_day)
    for day in missing_day:
        i = missing_day.index(day)
        try:
            ser[day] = ser[[before[i],after[i]]].mean()
        except:
            print('Can not apply fill_Missing_days1')
            return
    ser.sort_index(inplace=True)
    return ser

In [ ]:
stname2code = {"Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
    "Idaho": "ID", "Illinois": "IL", "Indiana": "IN",
    "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY",
    "Louisiana": "LA", "Maine": "ME", "Maryland": "MD","Massachusetts": "MA",
    "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO", "Montana": "MT",
    "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC",
    "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR",
    "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN",
    "Texas": "TX", "Utah": "UT", "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI",
    "Wyoming": "WY", "District of Columbia": "DC",
    "American Samoa": "AS", "Guam": "GU", "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR", "United States Minor Outlying Islands": "UM", "U.S. Virgin Islands": "VI"}
stcode2fip = {
    'WA': '53', 'DE': '10', 'DC': '11', 'WI': '55', 'WV': '54', 'HI': '15',
    'FL': '12', 'WY': '56', 'PR': '72', 'NJ': '34', 'NM': '35', 'TX': '48',
    'LA': '22', 'NC': '37', 'ND': '38', 'NE': '31', 'TN': '47', 'NY': '36',
    'PA': '42', 'AK': '02', 'NV': '32', 'NH': '33', 'VA': '51', 'CO': '08',
    'CA': '06', 'AL': '01', 'AR': '05', 'VT': '50', 'IL': '17', 'GA': '13',
    'IN': '18', 'IA': '19', 'MA': '25', 'AZ': '04', 'ID': '16', 'CT': '09',
    'ME': '23', 'MD': '24', 'OK': '40', 'OH': '39', 'UT': '49', 'MO': '29',
    'MN': '27', 'MI': '26', 'RI': '44', 'KS': '20', 'MT': '30', 'MS': '28',
    'SC': '45', 'KY': '21', 'OR': '41', 'SD': '46', 'AS': '60', 'GU': '66',
    'MP': '69', 'VI': '78'}
fip2stcode = {fip:stcode for stcode, fip in stcode2fip.items()}
extra_fip2county  = {'46102': 'Oglala Lakota', '02158': 'Kusilvak Census Area'}

def getCounties():
    "Function to return a dict of FIPS codes (keys) of U.S. counties (values)"
    d = {}
    r = requests.get("https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt", timeout=30)
    r.raise_for_status()
    reader = csv.reader(r.text.splitlines(), delimiter=',')    
    for line in reader:
        d[line[1] + line[2]] = line[3].replace(" County","")    
    return d
fip2county = getCounties()
fip2county.update(extra_fip2county)

from analysis_utils import formalize_fip
def fip_to_county(fip):
    if len(fip)==5:
        return fip2county[fip]
    else:
        return np.nan

In [ ]:
# Global Function -- transform
from analysis_utils import transform_mobilitydata


In [ ]:
# Test transform_mobilitydata
dtspp = pd.read_csv('./Delta TSPP/DTSPP_US_Mobility_formalized.csv',parse_dates=[4])
dtspp.fips = dtspp.fips.apply(formalize_fip)

transform_mobilitydata(dtspp, mob_feature_col='dtspp')

In [ ]:
# compare with origin df
dtspp

# Overview

In [ ]:
# url = 'https://raw.githubusercontent.com/GMU-GGS-NSF-ABM-Research/Mobility-Trends/master/Delta_MoPE.csv'
# dtspp = pd.read_csv(url)
dtspp = pd.read_csv('./Delta TSPP/DTSPP_US_Mobility_formalized.csv',parse_dates=[4])
dtspp.fips = dtspp.fips.apply(formalize_fip)
dtspp

In [ ]:
# to see day range
# google.groupby(by=['sub_region_1','sub_region_2']).apply(lambda df: (list(df.date)[0], list(df.date)[-1])) 
google =  pd.read_csv('./Google/Google_US_Mobility_formalized.csv', parse_dates=[4])
google.fips = google.fips.apply(formalize_fip)
google

In [ ]:
# url = 'https://raw.githubusercontent.com/descarteslabs/DL-COVID-19/master/DL-us-mobility-daterow.csv'
# dl = pd.read_csv(url)
dl = pd.read_csv('./Descartes Labs/DL_US_Mobility_formalized.csv', parse_dates=[4])
dl.fips = dl.fips.apply(formalize_fip)
dl

In [ ]:
# Add missing data

'''
before = [49+i*273 for i in range(2496)]
before.extend([87+i*273 for i in range(2496)])
before.extend([218+i*273 for i in range(2496)])

after = [50+i*273 for i in range(2496)]
after.extend([88+i*273 for i in range(2496)])
after.extend([219+i*273 for i in range(2496)])

before_dl = dl.loc[before]
after_dl = dl.loc[after]
before_dl.date = before_dl.date+pd.Timedelta(days=1)
after_dl.date = after_dl.date-pd.Timedelta(days=1)
before_dl = before_dl.reset_index(drop=True)
after_dl = after_dl.reset_index(drop=True)

missing_data[['m50','m50_index','pct_change']] = (before_dl[['m50','m50_index','pct_change']]+after_dl[['m50','m50_index','pct_change']])/2
missing_data
'''

# dl = dl.append(missing_data, ignore_index=True)
# dl = dl.sort_values(by=['admin1','admin2','date']).reset_index(drop=True)
# dl

In [ ]:
# facebook_global = pd.read_csv('./Facebook/movement-range-data-2020-03-01--2020-12-31.txt',sep='\t')
facebook = pd.read_csv('./Facebook/Facebook_US_Mobility_formalized.csv', parse_dates=[4])
facebook.fips = facebook.fips.apply(formalize_fip)
facebook

In [ ]:
# gov = pd.read_csv('./Gov/Trips_by_Distance.csv')
gov = pd.read_csv('./Gov/Gov_US_Mobility_formalized.csv', parse_dates=[4])
gov.fips = gov.fips.apply(formalize_fip)
gov

# Policies

In [ ]:
poly = pd.read_csv('./Policy/Chicago/Local-Policy-Responses-formalized.csv')
poly.fips = poly.fips.apply(formalize_fip)
poly

In [ ]:
poly.admin_level.value_counts()

# Visualiztion

In [ ]:
%%time
# Load data
dtspp = pd.read_csv('./Delta TSPP/DTSPP_US_Mobility_formalized.csv',parse_dates=[4])
dtspp.fips = dtspp.fips.apply(formalize_fip)

google =  pd.read_csv('./Google/Google_US_Mobility_formalized.csv', parse_dates=[4])
google.fips = google.fips.apply(formalize_fip)

dl = pd.read_csv('./Descartes Labs/DL_US_Mobility_formalized.csv', parse_dates=[4])
dl.fips = dl.fips.apply(formalize_fip)

facebook = pd.read_csv('./Facebook/Facebook_US_Mobility_formalized.csv', parse_dates=[4])
facebook.fips = facebook.fips.apply(formalize_fip)

gov = pd.read_csv('./Gov/Gov_US_Mobility_formalized.csv', parse_dates=[4])
gov.fips = gov.fips.apply(formalize_fip)

poly = pd.read_csv('./Policy/Chicago/Local-Policy-Responses-formalized.csv')
poly.fips = poly.fips.apply(formalize_fip)
len(poly)

measures = [['dtspp'],
 ['retail_and_recreation_percent_change_from_baseline', 'grocery_and_pharmacy_percent_change_from_baseline', 'parks_percent_change_from_baseline', 'transit_stations_percent_change_from_baseline', 'workplaces_percent_change_from_baseline', 'residential_percent_change_from_baseline'],
 ['m50_pct_change'],
 ['all_day_ratio_none_single_tile_users', 'all_day_bing_tiles_visited_relative_change'],
 ['Population Not Staying at Home', 'Number of Trips']]

mobility_data_sets = [dtspp, google, dl, facebook, gov]

In [ ]:
def integrate_info(fip, start_day='2020-03-01', end_day='2020-09-01'):
    daily_mobs = []
    errors = []
    if isinstance(fip,int):
        fip = '{:05d}'.format(fip)
        
    for i in range(5):
        df = mobility_data_sets[i]
        try:
            mob_measures = measures[i].copy()
            mob_measures.append('date')
            daily_mob = df[(df.fips==fip)&(df.date>=start_day)&(df.date<=end_day)].loc[:,mob_measures].set_index('date',drop=True)
            structure = pd.DataFrame(index = pd.date_range(start_day, end_day), columns=daily_mob.columns)
            daily_mob = daily_mob.reindex_like(structure).interpolate(method='linear', limit_direction='forward', limit=5,axis=0)
            daily_mobs.append(daily_mob)
        except:
            errors.append(i)
    return daily_mobs, errors
def display_policy(fip_tuple):
    
    if isinstance(fip_tuple[0],int):
            fips = ['{:05d}'.format(fip) for fip in fip_tuple]
    else:
            fips = [fip for fip in fip_tuple]
            
    select_poly = poly[poly.fips.apply(lambda fip: fip in fips)][['admin_level', 'state', 'countyname', 'fips', 'cityname', 'stsipstart',
       'stsipend', 'localsipstart', 'localsipend', 'stbusclose', 'localbusclose', 'stbusopen',
       'localbusopen', 'stresclose','stresopen', 'localresclose', 'localresopen']]

    return select_poly
def plot_df(df, figsize=(15, 6), y_label='Not given', title='Not given'):
    title_size = 5
    ax = df.plot(linewidth=2, figsize=figsize, xlabel=''); 

    # ticks 
    plt.xticks(fontweight="bold", fontproperties='Times New Roman');
    plt.yticks(fontweight="bold", fontproperties='Times New Roman'); 
    plt.ylabel(y_label, fontsize=15 ,fontweight="bold", fontproperties='Times New Roman');

    # title, legend & grid
    if title != 'Not given':
        plt.title(title, fontproperties='Times New Roman', 
              fontsize=title_size, fontweight="bold");
    plt.legend(prop={'family':'Times New Roman', 'weight':'bold'});
    plt.grid(visible=True);
    
    return ax
def plot_df_2yaxis(df, figsize=(15, 6), unite=['A', 'B'], title='Not given'):

    col1 = df.columns[0]
    col2 = df.columns[1]
    
    ax1 = df.plot(y=col1, linewidth=2, figsize=figsize, xlabel=''); 
   
    plt.xticks(fontweight="bold", fontproperties='Times New Roman');
    plt.yticks(fontweight="bold", fontproperties='Times New Roman'); 
    plt.ylabel(unite[0], fontsize=15, fontweight="bold", fontproperties='Times New Roman');
    ax1.legend(loc=0, prop={'family':'Times New Roman', 'weight':'bold'});

    ax2 = df[col2].plot(secondary_y=True)
    plt.yticks(fontweight="bold", fontproperties='Times New Roman');
    plt.ylabel(unite[1], fontsize=15, fontweight="bold", fontproperties='Times New Roman');
    ax2.legend(loc=4, prop={'family':'Times New Roman', 'weight':'bold'});
    
    # title, legend & grid
    if title != 'Not given':
        plt.title(title, fontproperties='Times New Roman', 
              fontsize=title_size, fontweight="bold");
    
    plt.grid(visible=True);
    
    return (ax1,ax2)
def plot_all_mobility(fip):
    if isinstance(fip,int):
        fip = '{:05d}'.format(fip) 
    daily_mobs, errors = integrate_info(fip)
    print('Error list:')
    display(errors)
    display_policy(fip)

    plot_df(daily_mobs[0], y_label='Delta TSPP: non-home time\nminute-change');
    plot_df(daily_mobs[1], y_label='Google: non-home time\npct change %');
    plot_df(daily_mobs[2], y_label='Descartes Labs:\nmax distance pct change %');
    plot_df_2yaxis(daily_mobs[3], unite=['Facebook:\nProp people not stay put (decimal)', 
                                     'Num visited places\npct change (decimal)']);
    plot_df_2yaxis(daily_mobs[4], unite=['Gov: Pop not at home', 'Num trips']);
def compare_dtspp(fip_tuple, start_day='2020-03-01', end_day='2020-09-01', display_policy=False, plot_pic=False):
    fips = [formalize_fip(fip) for fip in fip_tuple]
    if display_policy:
        policies = poly if 'fips' in poly.columns else poly.reset_index()
        display(policies.loc[policies.fips.isin(fips)])
    result = transform_mobilitydata(dtspp.loc[dtspp.fips.isin(fips)],
                                   start_day=start_day, end_day=end_day)
    if plot_pic:
        result.plot(figsize=(15, 6))
    return result


In [ ]:
# test function
plot_all_mobility(42003)

In [ ]:
# test function
structure = compare_dtspp((1001,1003,5001,19047), plot_pic=True)

In [ ]:
# display all region with policy
fips_lst = list(poly.fips.value_counts().index)
fips_lst.sort()
display(len(fips_lst))
display(fips_lst)

In [ ]:
# region with policy and dtspp mobility
fips_lst = list(set(poly.fips).intersection(set(dtspp.fips)))
fips_lst.sort()
display(len(fips_lst))
display(fips_lst)

## Mobility & Stay-at-home Order

In [ ]:
%%time
non_sip_fip = tuple(set(poly[(poly.stsipstart.isna()) & (poly.localsipstart.isna())].fips))
with_sip_fip = tuple(set(poly[(~poly.stsipstart.isna()) | (~poly.localsipstart.isna())].fips))
mob_non_sip = compare_dtspp(non_sip_fip)
mob_with_sip = compare_dtspp(with_sip_fip)

In [ ]:
mean_mob_with_sip = mob_with_sip.mean(axis=1)
mean_mob_non_sip = mob_non_sip.mean(axis=1)
compare_dtspp_ = pd.DataFrame()
compare_dtspp_['mean_mob_with_sip'] = mean_mob_with_sip
compare_dtspp_['mean_mob_non_sip'] = mean_mob_non_sip

compare_dtspp_.plot(figsize=(15,6), fontsize=15);

regions with stay-at-home policies tend to have lower mobilities.

## Mobility & Policy Type Counts

In [ ]:
poly_count = poly[['state','countyname','fips','cityname','stsipstart',
                   'localsipstart','stbusclose', 'localbusclose', 'stresclose',
                   'localresclose']]
poly_count['polycount'] = 6-poly_count[['stsipstart','localsipstart','stbusclose', 'localbusclose', 'stresclose',
            'localresclose']].isna().sum(axis=1)

In [ ]:
poly_count

In [ ]:
fip_by_polycount = poly_count.groupby(by='polycount').apply(lambda df: set(df.fips))
display(fips_by_polycount)

In [ ]:
compare_dtspp_by_polycount = pd.DataFrame()
for i in range(7):
    col_name = 'mean_mob_polycount_%d'%i
    mean_mob = compare_dtspp(tuple(fip_by_polycount[i]), display_policy=False).mean(axis=1)
    compare_dtspp_by_polycount[col_name] = mean_mob

display(compare_dtspp_by_polycount)
compare_dtspp_by_polycount.plot(figsize=(15,6), fontsize=15);

regions with more close-related policies tend to have lower mobilities.

In [ ]:
compare_dtspp_by_polycount.mean().plot(style="o-", ms=7, figsize=(15,6), fontsize=15);
plt.xticks(rotation=10);

# Metro to County Match

In [ ]:
# Locate folder
os.chdir(workspace('mobility/trends'))     

In [ ]:
df1 = pd.read_csv('google-trend-geocode2metro.csv',index_col=0)[['metro']]
df2 = pd.read_csv('county-msa-csa.csv', encoding_errors='ignore')

fit_metro = set(df2['MSA Title']).intersection(set(df1.metro)) # metro in both df1 and df2
complement = set(df2['MSA Title']) - (set(df1.metro))          # metro in df2 but has no fit in df1
match_dic = { x : set(re.split(r'[-\s.]+',x)) for x in complement if isinstance(x, str)} # metro split words

df1['msa_title'] = np.nan
df1['msa_code'] = np.nan
df1['county_code'] = np.nan

In [ ]:
print('Fit Metro:',len(fit_metro))
print(fit_metro)

In [ ]:
# match metro in both df1 and df2
def complete_fit_metro(ob):
    if ob.metro in fit_metro:
        ob.county_code = ', '.join([formalize_fip(x) for x in 
                                    list(df2[df2['MSA Title'] == ob.metro]
                                         ['County Code'].values)])
        ob.msa_code = df2[df2['MSA Title'] == ob.metro]['MSA Code'].values[0]
        ob.msa_title = df2[df2['MSA Title'] == ob.metro]['MSA Title'].values[0]
    return ob

In [ ]:
# apply function
df1 = df1.apply(complete_fit_metro,axis=1)

In [ ]:
# match df2 metros to df1
def match_metro_to_msa(testword):
    test_word_set = set(re.split(r'[-\s.]+',testword))

    # match state
    states = {x for x in test_word_set if len(x)==2 and x.isupper() }
    temp_match_dic_st = {key:match_dic[key] for key in match_dic.keys() if len(states.intersection(match_dic[key]))>0}

    # match metro
    extra_filter_dic = {'San','Santa','St', 'Springs', 'City','Fort'}
    test_word_set_2 = {word for word in test_word_set if word not in states and word not in extra_filter_dic}
    temp_match_dic_2 = { key : {w for w in temp_match_dic_st[key] if w not in states and w not in extra_filter_dic
                               } for key in temp_match_dic_st.keys()}
    
    # Calculate average match similarity
    match_result = {key:
                round(( (len(test_word_set_2.intersection(temp_match_dic_2[key]))/len(test_word_set_2)) 
                + (len(test_word_set_2.intersection(temp_match_dic_2[key]))/len(temp_match_dic_2[key])))/2 , 3)
                for key in temp_match_dic_2.keys()}
    filter1 = [item for item in sorted(match_result.items(), key=lambda x: x[1], reverse=True) if item[1]>0.35]
    result = ', '.join([item[0] for item in filter1])
    return result

In [ ]:
# add rest info
def add_rest_info(ob):
    if not isinstance(ob.msa_title,str):
        
        msa_lst = match_metro_to_msa(ob.metro).split(', ')
        
        if len(msa_lst) > 0:
            try:
                ob.msa_title = match_metro_to_msa(ob.metro)
                ob.msa_code = ', '.join([df2[df2['MSA Title'] == msa]['MSA Code'].values[0] for msa in msa_lst])
                ob.county_code = ', '.join(list(set([formalize_fip(x) 
                                for msa in msa_lst 
                                for x in df2[df2['MSA Title'] == msa]['County Code'].values])))
            except:
                ob.msa_title = np.nan
                ob.msa_code = np.nan
                ob.county_code = np.nan
                
    return ob

In [ ]:
# apply function
df1 = df1.apply(add_rest_info,axis=1)

In [ ]:
df1.head(40)

## Check Match

In [ ]:
def check(x):
    if isinstance(x,str):
        return len(x.split(', ')) 
    else:
        return np.nan

df1[df1.msa_code.apply(check)==2]

In [ ]:
index = 'US-FL-548'
display(df1.loc[index])
display(df1.loc[index].metro)
display(df1.loc[index].msa_title)

In [ ]:
# adjust function match_metro_to_msa(testword)
extra_filter_dic = {'San','Santa','St', 'Springs', 'City','Fort'} 

In [ ]:
# drop nan
df1 = df1.dropna()

## Data Could be Used

In [ ]:
# Useble fips
metro_county = set(', '.join(list(df1.county_code)).split(', '))
print("All metro contains ",len(metro_county), ' counties')
print("DTSPP Data Set contains ",len(set(dtspp.fips)), ' counties')
print("Intersection contains ",len(set(dtspp.fips).intersection(metro_county)), ' counties')
print("Loss county fip ", metro_county-set(dtspp.fips))
display(df1[df1.county_code.str.contains('51515')])

In [ ]:
# drop
df1 = df1.drop('US-VA-573')

In [ ]:
df1

In [ ]:
# Save
df1.to_csv('googel-trend-metro2county-clean-version.csv')